# Phase 8 — model finalization and packaging

Regenerates only capped Phase 7 predictions, selects a safe generation policy, routes unresolved cases to hierarchical summarization, packages the accepted adapter, and verifies a clean five-domain reload. Attach the completed Phase 5, Phase 6 pilot, and Phase 7 notebook outputs. Use Kaggle GPU T4 x2.

In [ ]:
import gc
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

BRANCH = "refactoring/phase-transformer"
REPOSITORY = "https://github.com/SachinGupta2012/Text-Summarizer.git"
PROJECT_ROOT = Path("/kaggle/temp/text-summarizer/project")
OUTPUT_DIRECTORY = Path("/kaggle/working/text-summarizer/phase8/model-finalization-v1")

print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Output: {OUTPUT_DIRECTORY}")

In [ ]:
requirements = [
    "transformers>=4.57,<4.58",
    "accelerate>=1.10,<2",
    "bitsandbytes>=0.48,<1",
    "peft>=0.17,<0.19",
    "bert-score>=0.3.13,<0.4",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("Phase 8 dependencies are ready.")

In [ ]:
if not (PROJECT_ROOT / ".git").is_dir():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(PROJECT_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
ml_source = str(PROJECT_ROOT / "ml" / "src")
if ml_source not in sys.path:
    sys.path.insert(0, ml_source)
print(f"Project commit: {commit}")

In [ ]:
phase5_candidates = []
for path in sorted(Path("/kaggle/input").rglob("baseline-report.json")):
    if all(
        (path.parent / name).is_file()
        for name in ("benchmark-selection.jsonl", "base-predictions.jsonl", "bertscore.jsonl")
    ):
        phase5_candidates.append(path)
if len(phase5_candidates) != 1:
    raise RuntimeError(f"Expected one complete Phase 5 input; found {phase5_candidates}")
PHASE5_REPORT_PATH = phase5_candidates[0]
PHASE5_DIRECTORY = PHASE5_REPORT_PATH.parent

pilot_candidates = []
for path in sorted(Path("/kaggle/input").rglob("training-report.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("run_mode") == "pilot" and all(
        (path.parent / relative).is_file()
        for relative in (
            "COMPLETED",
            "adapter/adapter_model.safetensors",
            "adapter/adapter_config.json",
            "tokenizer/tokenizer_config.json",
        )
    ):
        pilot_candidates.append(path)
if len(pilot_candidates) != 1:
    raise RuntimeError(f"Expected one complete Phase 6 pilot input; found {pilot_candidates}")
PILOT_REPORT_PATH = pilot_candidates[0]
PILOT_DIRECTORY = PILOT_REPORT_PATH.parent
ADAPTER_DIRECTORY = PILOT_DIRECTORY / "adapter"
TOKENIZER_DIRECTORY = PILOT_DIRECTORY / "tokenizer"

phase7_candidates = []
for path in sorted(Path("/kaggle/input").rglob("full-comparison-report.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("decision") == "accept_pilot" and all(
        (path.parent / name).is_file()
        for name in ("pilot-predictions.jsonl", "pilot-bertscore.jsonl", "phase7-config.json")
    ):
        phase7_candidates.append(path)
if len(phase7_candidates) != 1:
    raise RuntimeError(f"Expected one accepted Phase 7 input; found {phase7_candidates}")
PHASE7_REPORT_PATH = phase7_candidates[0]
PHASE7_DIRECTORY = PHASE7_REPORT_PATH.parent
PHASE7_CONFIG_PATH = PHASE7_DIRECTORY / "phase7-config.json"

print(f"Phase 5 input: {PHASE5_DIRECTORY}")
print(f"Phase 6 pilot: {PILOT_DIRECTORY}")
print(f"Phase 7 evaluation: {PHASE7_DIRECTORY}")

In [ ]:
from summarizer_ml.evaluation import (
    Phase5Report,
    Phase7Decision,
    Phase7Report,
    Phase8Config,
    audit_predictions,
    capped_record_ids,
    load_benchmark_selection,
    load_predictions,
    sha256_file,
)

CONFIG = Phase8Config()
PHASE5_REPORT = Phase5Report.model_validate_json(PHASE5_REPORT_PATH.read_text(encoding="utf-8"))
PHASE7_REPORT = Phase7Report.model_validate_json(PHASE7_REPORT_PATH.read_text(encoding="utf-8"))
PILOT_REPORT = json.loads(PILOT_REPORT_PATH.read_text(encoding="utf-8"))
if PHASE7_REPORT.decision is not Phase7Decision.ACCEPT_PILOT:
    raise RuntimeError("Phase 8 requires an accepted Phase 7 pilot.")

SELECTION_PATH = PHASE5_DIRECTORY / "benchmark-selection.jsonl"
BASE_PREDICTIONS_PATH = PHASE5_DIRECTORY / "base-predictions.jsonl"
BASE_BERTSCORE_PATH = PHASE5_DIRECTORY / "bertscore.jsonl"
ORIGINAL_PREDICTIONS_PATH = PHASE7_DIRECTORY / "pilot-predictions.jsonl"
ORIGINAL_BERTSCORE_PATH = PHASE7_DIRECTORY / "pilot-bertscore.jsonl"
ADAPTER_PATH = ADAPTER_DIRECTORY / "adapter_model.safetensors"
selection = load_benchmark_selection(SELECTION_PATH)
base_predictions = load_predictions(BASE_PREDICTIONS_PATH)
original_predictions = load_predictions(ORIGINAL_PREDICTIONS_PATH)
if PHASE7_REPORT.adapter_sha256 != sha256_file(ADAPTER_PATH):
    raise RuntimeError("Phase 6 adapter does not match the accepted Phase 7 adapter.")

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
config_path = OUTPUT_DIRECTORY / "phase8-config.json"
config_payload = CONFIG.model_dump_json(indent=2) + "\n"
if config_path.is_file() and config_path.read_text(encoding="utf-8") != config_payload:
    raise RuntimeError("Existing Phase 8 output used a different configuration.")
config_path.write_text(config_payload, encoding="utf-8")
input_identity_path = OUTPUT_DIRECTORY / "phase8-inputs.json"
input_identity = {
    "benchmark_selection_sha256": sha256_file(SELECTION_PATH),
    "phase7_report_sha256": sha256_file(PHASE7_REPORT_PATH),
    "adapter_sha256": sha256_file(ADAPTER_PATH),
    "original_predictions_sha256": sha256_file(ORIGINAL_PREDICTIONS_PATH),
    "original_bertscore_sha256": sha256_file(ORIGINAL_BERTSCORE_PATH),
}
input_payload = json.dumps(input_identity, indent=2, sort_keys=True) + "\n"
if (
    input_identity_path.is_file()
    and input_identity_path.read_text(encoding="utf-8") != input_payload
):
    raise RuntimeError("Existing Phase 8 output belongs to different saved inputs.")
input_identity_path.write_text(input_payload, encoding="utf-8")
repair_ids = capped_record_ids(original_predictions, CONFIG.original_max_new_tokens)
repair_id_set = set(repair_ids)
repair_selection = tuple(item for item in selection if item.record_id in repair_id_set)
REPAIR_IDS_PATH = OUTPUT_DIRECTORY / "repair-record-ids.jsonl"
REPAIR_IDS_PATH.write_text(
    "".join(json.dumps({"record_id": record_id}) + "\n" for record_id in repair_ids),
    encoding="utf-8",
)
original_audit = audit_predictions(
    selection,
    original_predictions,
    CONFIG.original_max_new_tokens,
    OUTPUT_DIRECTORY,
    "preflight-original",
    CONFIG,
)
print(CONFIG.model_dump_json(indent=2))
print(f"Frozen benchmark records: {len(selection):,}")
print(f"Capped records selected for repair: {len(repair_ids):,}")
print(original_audit.model_dump_json(indent=2))

In [ ]:
import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Phase 8 requires Kaggle GPU T4 x2.")
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {properties.name}; memory={properties.total_memory / (1024**3):.1f} GiB")

In [ ]:
command = [
    sys.executable,
    "-m",
    "accelerate.commands.launch",
    "--num_processes",
    "2",
    "--num_machines",
    "1",
    "--mixed_precision",
    "fp16",
    "--multi_gpu",
    str(PROJECT_ROOT / "ml" / "scripts" / "generate_adapter_predictions.py"),
    "--phase5-report",
    str(PHASE5_REPORT_PATH),
    "--selection",
    str(SELECTION_PATH),
    "--adapter-directory",
    str(ADAPTER_DIRECTORY),
    "--tokenizer-directory",
    str(TOKENIZER_DIRECTORY),
    "--phase7-config",
    str(PHASE7_CONFIG_PATH),
    "--output-directory",
    str(OUTPUT_DIRECTORY),
    "--stage",
    "full",
    "--record-ids",
    str(REPAIR_IDS_PATH),
    "--max-new-tokens",
    str(CONFIG.candidate_max_new_tokens),
    "--inference-batch-size",
    str(CONFIG.inference_batch_size),
    "--checkpoint-subdirectory",
    "repair-workers",
]
print("Launching selective repair:", " ".join(command))
environment = os.environ.copy()
environment["PYTHONPATH"] = os.pathsep.join(
    value for value in [ml_source, environment.get("PYTHONPATH")] if value
)
subprocess.run(command, check=True, cwd=PROJECT_ROOT, env=environment)

In [ ]:
from summarizer_ml.evaluation import merge_prediction_shards, overlay_repairs

worker_paths = tuple(sorted(OUTPUT_DIRECTORY.glob("repair-workers/rank-*.jsonl")))
if len(worker_paths) != 2:
    raise RuntimeError(f"Expected two repair worker checkpoints; found {worker_paths}")
REPAIR_PREDICTIONS_PATH = OUTPUT_DIRECTORY / "repair-predictions.jsonl"
repair_predictions = merge_prediction_shards(
    repair_selection,
    worker_paths,
    REPAIR_PREDICTIONS_PATH,
)
CANDIDATE_PREDICTIONS_PATH = OUTPUT_DIRECTORY / "candidate-predictions.jsonl"
candidate_predictions = overlay_repairs(
    selection,
    original_predictions,
    repair_predictions,
    repair_ids,
    CANDIDATE_PREDICTIONS_PATH,
)
print(f"Repair predictions: {len(repair_predictions):,}")
print(f"Candidate prediction coverage: {len(candidate_predictions):,}")

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print("Inference workers exited; GPU cache cleared before selective BERTScore.")

In [ ]:
from bert_score import score as bert_score


def load_scores(path):
    if not path.is_file():
        return {}
    return {
        row["record_id"]: float(row["f1"])
        for row in (
            json.loads(line)
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        )
    }


base_bertscore = load_scores(BASE_BERTSCORE_PATH)
original_bertscore = load_scores(ORIGINAL_BERTSCORE_PATH)
REPAIR_BERTSCORE_PATH = OUTPUT_DIRECTORY / "repair-bertscore.jsonl"
repair_bertscore = load_scores(REPAIR_BERTSCORE_PATH)
selection_by_id = {item.record_id: item for item in selection}
candidate_by_id = {item.record_id: item for item in candidate_predictions}
missing_ids = [record_id for record_id in repair_ids if record_id not in repair_bertscore]
if missing_ids:
    references = [selection_by_id[record_id].reference_summary for record_id in missing_ids]
    candidates = [candidate_by_id[record_id].prediction for record_id in missing_ids]
    _, _, f1 = bert_score(
        candidates,
        references,
        model_type=PHASE5_REPORT.config.bertscore_model_id,
        lang="en",
        batch_size=8,
        device="cuda:0",
        rescale_with_baseline=True,
        verbose=True,
    )
    repair_bertscore.update(dict(zip(missing_ids, map(float, f1.tolist()), strict=True)))
temporary = REPAIR_BERTSCORE_PATH.with_name(f"{REPAIR_BERTSCORE_PATH.name}.tmp")
with temporary.open("w", encoding="utf-8", newline="\n") as output:
    for record_id in repair_ids:
        output.write(json.dumps({"record_id": record_id, "f1": repair_bertscore[record_id]}) + "\n")
temporary.replace(REPAIR_BERTSCORE_PATH)
candidate_bertscore = dict(original_bertscore)
candidate_bertscore.update(repair_bertscore)
CANDIDATE_BERTSCORE_PATH = OUTPUT_DIRECTORY / "candidate-bertscore.jsonl"
temporary = CANDIDATE_BERTSCORE_PATH.with_name(f"{CANDIDATE_BERTSCORE_PATH.name}.tmp")
with temporary.open("w", encoding="utf-8", newline="\n") as output:
    for item in selection:
        output.write(
            json.dumps({"record_id": item.record_id, "f1": candidate_bertscore[item.record_id]})
            + "\n"
        )
temporary.replace(CANDIDATE_BERTSCORE_PATH)
print(f"Repair BERTScore records added: {len(missing_ids):,}")
print(f"Candidate BERTScore coverage: {len(candidate_bertscore):,}")

In [ ]:
from summarizer_ml.evaluation import finalize_phase8

result = finalize_phase8(
    selection,
    base_predictions,
    original_predictions,
    candidate_predictions,
    base_bertscore,
    original_bertscore,
    candidate_bertscore,
    PHASE7_REPORT,
    OUTPUT_DIRECTORY,
    CONFIG,
    benchmark_selection_path=SELECTION_PATH,
    base_predictions_path=BASE_PREDICTIONS_PATH,
    adapter_path=ADAPTER_PATH,
    phase7_report_path=PHASE7_REPORT_PATH,
    model_id=PHASE5_REPORT.config.model_id,
)
print(result.model_dump_json(indent=2))
print(f"Selected generation policy: {result.policy_decision.value}")

In [ ]:
from summarizer_ml.evaluation import create_model_package

PACKAGE_DIRECTORY = OUTPUT_DIRECTORY / "qwen3-summarizer-adapter-v1"
ARCHIVE_PATH = OUTPUT_DIRECTORY / "qwen3-summarizer-adapter-v1.tar.gz"
package_manifest = create_model_package(
    ADAPTER_DIRECTORY,
    TOKENIZER_DIRECTORY,
    OUTPUT_DIRECTORY / "generation-policy.json",
    OUTPUT_DIRECTORY / "phase8-final-report.json",
    PACKAGE_DIRECTORY,
    ARCHIVE_PATH,
    model_id=PHASE5_REPORT.config.model_id,
    model_revision=PILOT_REPORT["model_revision"],
)
print(package_manifest.model_dump_json(indent=2))
print(f"Package archive: {ARCHIVE_PATH}; bytes={ARCHIVE_PATH.stat().st_size:,}")

In [ ]:
gc.collect()
torch.cuda.empty_cache()
policy = json.loads((OUTPUT_DIRECTORY / "generation-policy.json").read_text(encoding="utf-8"))
datasets = sorted({item.dataset for item in selection})
smoke_selection = tuple(
    min((item for item in selection if item.dataset == dataset), key=lambda item: item.record_id)
    for dataset in datasets
)
SMOKE_IDS_PATH = OUTPUT_DIRECTORY / "package-smoke-record-ids.jsonl"
SMOKE_IDS_PATH.write_text(
    "".join(json.dumps({"record_id": item.record_id}) + "\n" for item in smoke_selection),
    encoding="utf-8",
)
smoke_command = [
    sys.executable,
    str(PROJECT_ROOT / "ml" / "scripts" / "generate_adapter_predictions.py"),
    "--phase5-report",
    str(PHASE5_REPORT_PATH),
    "--selection",
    str(SELECTION_PATH),
    "--adapter-directory",
    str(PACKAGE_DIRECTORY),
    "--tokenizer-directory",
    str(PACKAGE_DIRECTORY),
    "--phase7-config",
    str(PHASE7_CONFIG_PATH),
    "--output-directory",
    str(OUTPUT_DIRECTORY),
    "--stage",
    "full",
    "--record-ids",
    str(SMOKE_IDS_PATH),
    "--max-new-tokens",
    str(policy["max_new_tokens"]),
    "--inference-batch-size",
    "1",
    "--checkpoint-subdirectory",
    "package-smoke-workers",
]
print("Launching clean package reload smoke test...")
subprocess.run(smoke_command, check=True, cwd=PROJECT_ROOT, env=environment)
SMOKE_PREDICTIONS_PATH = OUTPUT_DIRECTORY / "package-smoke-predictions.jsonl"
smoke_predictions = merge_prediction_shards(
    smoke_selection,
    (OUTPUT_DIRECTORY / "package-smoke-workers" / "rank-00.jsonl",),
    SMOKE_PREDICTIONS_PATH,
)
if len(smoke_predictions) != len(datasets) or any(
    not item.prediction.strip() for item in smoke_predictions
):
    raise RuntimeError("Packaged adapter smoke test failed.")
SMOKE_REPORT_PATH = OUTPUT_DIRECTORY / "package-smoke-report.json"
SMOKE_REPORT_PATH.write_text(
    json.dumps(
        {
            "records": len(smoke_predictions),
            "datasets": datasets,
            "all_nonempty": True,
            "package_adapter_sha256": package_manifest.adapter_sha256,
        },
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)
print(f"Clean package reload passed across: {', '.join(datasets)}")

In [ ]:
required = [
    OUTPUT_DIRECTORY / "phase8-config.json",
    OUTPUT_DIRECTORY / "phase8-inputs.json",
    OUTPUT_DIRECTORY / "phase8-final-report.json",
    OUTPUT_DIRECTORY / "generation-policy.json",
    OUTPUT_DIRECTORY / "final-predictions.jsonl",
    OUTPUT_DIRECTORY / "final-bertscore.jsonl",
    OUTPUT_DIRECTORY / "chunking-route.jsonl",
    OUTPUT_DIRECTORY / "qualitative-audit.jsonl",
    PACKAGE_DIRECTORY / "adapter_model.safetensors",
    PACKAGE_DIRECTORY / "model-manifest.json",
    ARCHIVE_PATH,
    SMOKE_REPORT_PATH,
]
for path in required:
    print(
        path.relative_to(OUTPUT_DIRECTORY),
        "exists:",
        path.is_file(),
        "bytes:",
        path.stat().st_size if path.exists() else 0,
    )
    if not path.is_file():
        raise RuntimeError(f"Missing Phase 8 artifact: {path}")
print("Phase 8 finalization, packaging, and clean-reload gates passed.")